### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="miami_housing",
    dataset_year="2016", # data from 2016, curated in 2021
    domain_str="finance",
    # Data Source
    dataset_source="Kaggle",
    # We were unable to find the true original source from the paper, but this is the oldest available data source.
    original_dataset_source_download_link="https://www.openml.org/d/43093",
    download_description="""
wget https://www.openml.org/data/download/22047757/miami2016.arff && \
mkdir -p local-data-warehouse/miami_housing && \
mv miami2016.arff local-data-warehouse/miami_housing/
""",
    # References
    academic_reference_bibtex="""@article{bourassa2021big,
  title={Big data, accessibility and urban house prices},
  author={Bourassa, Steven C and Hoesli, Martin and Merlin, Louis and Renne, John},
  journal={Urban Studies},
  volume={58},
  number={15},
  pages={3176--3195},
  year={2021},
  publisher={SAGE Publications Sage UK: London, England}
}
""",
    academic_reference_bibtex_key="bourassa2021big",
    license="CC BY-NC-SA",
    data_tags=["IID", "Spatial"],
    curation_comments="""
- We log scale the target.
- We drop duplicated homes (non-unique identifiers and location) (~1% of the data).
- We drop the ID column "PARCELNO".
- Anomaly: while the original paper describes data with 57k samples, we only have 13k on OpenML or Kaggle.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="SALE_PRC",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [ ]:
import pandas as pd
import arff
import numpy as np

with open(f"{dataset_mold.path}/miami2016.arff") as f:
    data = arff.load(f)

df = pd.DataFrame(data["data"], columns=[x[0] for x in data["attributes"]])
df = df.drop_duplicates(subset=["PARCELNO"])
df = df.drop(columns=["PARCELNO"])

cat_features = [
    "avno60plus"
]
df[cat_features] = df[cat_features].astype("category")
df[task_mold.target_column_name] = np.log(df[task_mold.target_column_name])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()